In [ ]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

#Load the dataset
df = pd.read_csv("expected_ctc.csv")

#Preprocessing
#Drop unique identifiers
df_model = df.drop(['IDX', 'Applicant_ID', "Passing_Year_Of_PHD","Passing_Year_Of_PG",
    "Curent_Location", "Preferred_location","Passing_Year_Of_Graduation",
    "Organization", "University_Grad", "University_PG", "University_PHD"], axis=1)

#Handle missing values
cat_cols = df_model.select_dtypes(include=['object', 'string']).columns
df_model[cat_cols] = df_model[cat_cols].fillna('Unknown')

# Binary encoding for Inhand_Offer
df_model['Inhand_Offer'] = df_model['Inhand_Offer'].map({'Y': 1, 'N': 0, 'Unknown': 0})

# Label Encode categorical features
label_encoders = {}
for col in cat_cols:
    if col == 'Inhand_Offer': continue
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col].astype(str))
    label_encoders[col] = le

#Split Data
X = df_model.drop('Expected_CTC', axis=1)
y = df_model['Expected_CTC']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#Train the model (Random Forest)
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

#Evaluate
y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Model Performance:")
print(f"RMSE: {rmse:.2f}")
print(f"R2 Score: {r2:.4f}")

#Predicting for a new applicant
def predict_salary(input_data):
    input_df = pd.DataFrame([input_data])
    return model.predict(input_df)
#Print Actual vs Predicted CTC Comparison
comparison_df = pd.DataFrame({
    'Actual_CTC': y_test.values,
    'Predicted_CTC': y_pred.astype(int) 
})

#Display the first 10 rows

print(comparison_df.head(10))

#Column for the difference (Error)
comparison_df['Difference'] = comparison_df['Actual_CTC'] - comparison_df['Predicted_CTC']
print("\nSummary of Differences:")
print(comparison_df.head(10))



Model Performance:
RMSE: 23568.23
R2 Score: 0.9996
   Actual_CTC  Predicted_CTC
0     3113073        3130883
1     1263089        1250764
2     3638037        3601988
3     2759980        2759198
4     1040303        1061879
5     2208350        2207213
6     1281873        1290140
7     2322435        2331973
8     3219206        3261344
9      650182         616685

Summary of Differences:
   Actual_CTC  Predicted_CTC  Difference
0     3113073        3130883      -17810
1     1263089        1250764       12325
2     3638037        3601988       36049
3     2759980        2759198         782
4     1040303        1061879      -21576
5     2208350        2207213        1137
6     1281873        1290140       -8267
7     2322435        2331973       -9538
8     3219206        3261344      -42138
9      650182         616685       33497
